
## Scheduling and Monitoring Your Production Pipeline

### Scheduling Strategy

1. **Fixed Schedule (Already Configured)**: Your job is scheduled to run at **6:00 AM** daily. Ensure the schedule is set to the correct **timezone** (e.g., UTC or your local zone) to avoid ambiguity.

2. **Trigger Rules for Task Dependencies**:
   - **Bronze Ingestion** → retry **2 times** on failure (configured ✅).
   - **Silver Transform, Gold Model, Data Quality Checks** → retry **1 time** on failure (configured ✅).
   - Each downstream task is set to trigger only after the upstream task completes successfully (`depends_on` / task dependency chain).

3. **Parameterized Catalog**: The `catalog` parameter you added allows the pipeline to run against different catalogs (e.g., dev, staging, prod) without modifying the job definition. In production, pass the production catalog name.

4. **Consider Adding**:
   - **Max concurrent runs** set to 1 (avoid overlapping runs).
   - **Alert on stale job** — trigger an alert if the job hasn't started within a certain time window (e.g., if it hasn't run by 7:00 AM, something may be wrong with the scheduler).
   - **Retry with backoff** — use exponential backoff between retries to handle transient failures gracefully.

---

### Monitoring Strategy

1. **Email Notifications (Already Configured)**: You will receive an email if the job fails (configured ✅). Make sure the email addresses are up to date and include a shared team inbox or on-call rotation.

2. **Job Run History**: Regularly check the **Runs tab** in the Databricks Jobs UI to review:
   - Success/failure status of each run
   - Duration trends (is the pipeline getting slower?)
   - Retry counts (are retries happening frequently — indicating an upstream issue?)

3. **Task-Level Monitoring**: Since each task (bronze → silver → gold → DQ checks) is separate, you can identify **exactly which stage failed** without digging through logs.

4. **Data Quality Checks (Final Task)**: The last task in the chain validates data quality. Monitor:
   - DQ rule pass/fail rates over time
   - Row counts and schema drift between layers
   - If DQ checks fail, the gold layer may have incomplete or incorrect data

5. **System Tables for Proactive Monitoring**: Use Databricks **system tables** for deeper, queryable insights:
   - `system.lakeflow.job_runs` — run history, durations, and failure counts across all runs
   - `system.compute.clusters` — cluster utilization and autoscaling behavior
   - Set up **SQL alerts** on these tables (e.g., alert if the latest run failed or if run duration exceeds a threshold).

6. **Logging and Alerting Best Practices**:
   - Forward job failure emails to a paging system (PagerDuty, Opsgenie, etc.) for on-call response.
   - Add a **stale run alert** — notify if no successful run has occurred in the last 24 hours.
   - Log key metrics (row counts, latency, DQ failures) to a Delta table for trend analysis and dashboarding.

7. **Recovery on Failure**:
   - On Bronze failure (after 2 retries): check source system availability, network/connectivity, or schema changes.
   - On Silver/Gold failure (after 1 retry): check for upstream data issues, transformation logic errors, or resource constraints.
   - On DQ failure: investigate the specific quality rule that failed — it may indicate bad source data or a logic bug.
   - Use the **Repair Run** feature in Databricks to re-run from the failed task (not the entire pipeline) after fixing the issue.

---

### Summary Checklist

| Aspect | Status | Notes |
|---|---|---|
| Schedule (6:00 AM) | ✅ Configured | Verify timezone |
| Task dependencies | ✅ Configured | Bronze → Silver → Gold → DQ |
| Retries (Bronze: 2, Others: 1) | ✅ Configured | Consider backoff |
| Catalog parameter | ✅ Configured | Pass prod catalog in production |
| Email notifications | ✅ Configured | Route to on-call / paging |
| Max concurrent runs | ⬜ Recommended | Set to 1 |
| Stale run alert | ⬜ Recommended | Alert if no run within expected window |
| SQL alerts on system tables | ⬜ Recommended | Proactive monitoring |
| Repair run workflow | ⬜ Recommended | Re-run from failed task, not full pipeline |